# Regression

In [ ]:
import os
import logging
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import shap

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split, RepeatedKFold, GridSearchCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBRegressor

In [ ]:
from utils.constants import RANDOM_SEED
from utils.common import (
    get_data_folder_path,
    set_plotting_config,
    plot_boxplot_by_class,
    plot_correlation_matrix,
)
from utils.evals import (
    describe_input_features,
    compute_regression_metrics,
    build_coefficients_table,
    plot_coefficients_values,
    plot_coefficients_significance,
    plot_eval_metrics_xgb,
    plot_gain_metric_xgb,
    plot_shap_importance,
    plot_shap_beeswarm,
)
from utils.feature_selection import run_feature_selection_steps

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

pd.set_option("display.max_columns", None)
pd.options.display.float_format = "{:,.2f}".format

mpl.rcParams["font.sans-serif"] = "Arial"
plt.set_loglevel("WARNING")

# plots configuration
sns.set_style("darkgrid")
sns.set_palette("colorblind")
set_plotting_config()
%matplotlib inline

## 1. Load Data

In this notebook, we will use the **Medical Insurance Payout Dataset**. This dataset contains historical data for over 1300 insurance customers (age, sex, BMI, number of children, smoking habits, and region) along with their actual medical charges. i.e., the expenditure for the customer (target variable).

Sources:
1. Kaggle: https://www.kaggle.com/datasets/harshsingh2209/medical-insurance-payout
2. Original source: https://raw.githubusercontent.com/JovianML/opendatasets/master/data/medical-charges.csv

In [ ]:
data_path = get_data_folder_path()

df_input = pd.read_csv(os.path.join(data_path, "expenses.csv"))

In [ ]:
# convert categorical columns into numerical
df_input["is_male"] = (df_input["sex"] == "male").astype(np.int8)
df_input["is_smoker"] = (df_input["smoker"] == "yes").astype(np.int8)
df_input = (
    pd.concat([
        df_input.drop(columns=["sex", "smoker", "region"]),
        pd.get_dummies(df_input["region"], prefix="region", dtype=np.int8)
    ], axis=1)
)

## 2. Process Data

### Target column

The target column is the medical charges for each customer. We want to build a model capable of predicting medical charges for new customers in order to help the insurance company to determine their pricing strategy.

In [ ]:
target_col = "charges"
test_size = 0.20

### Train test split

In [ ]:
df_input_train, df_input_test = train_test_split(
    df_input,
    test_size=test_size,
    random_state=RANDOM_SEED,
)

In [ ]:
describe_input_features(df_input, df_input_train, df_input_test)

### Scaling (Standardization)

In [ ]:
# Standardize training and test data
stdscaler = StandardScaler()

# training data
y_train = df_input_train[target_col]
X_train_all = (
    pd.DataFrame(
        # fit scaler on training data (and then transform training data)
        data=stdscaler.fit_transform(df_input_train),
        columns=df_input_train.columns,
        index=df_input_train.index
    )
    # remove target from the model input features table
    .drop(columns=[target_col])
)

# test data
y_test = df_input_test[target_col]
X_test_all = (
    pd.DataFrame(
        # use scaler fitted on training data to transform test data
        data=stdscaler.transform(df_input_test),
        columns=df_input_test.columns,
        index=df_input_test.index
    )
    # remove target from the model input features table
    .drop(columns=[target_col])
)

## 3. Exploratory Data Analysis (EDA)

### Boxplots by Target Class

In [ ]:
# use only training data to avoid bias in test results
df_boxplot = df_input_train.copy()

# get target quartiles
df_boxplot["charges_quartiles"] = pd.qcut(
    df_boxplot["charges"], q=4, labels=[f"Q{i}" for i in range(1, 5)],
)

In [ ]:
display(
    plot_boxplot_by_class(
        df_input=df_boxplot,
        class_col="charges_quartiles",
        plots_per_line=4,
        title="Features in input dataset by medical charges quartiles",
    )
)

### Pearson's Correlation

In [ ]:
display(
    plot_correlation_matrix(
        # use only training data to avoid bias in test results
        df=df_input_train, method="pearson", fig_height=10
    )
)

## 4. Feature Selection

In [ ]:
fs_steps = {
    "manual": {
        "cols_to_exclude": []
    },
    "null_variance": None,
    "correlation": {"threshold": 0.8},
    "vif": {"threshold": 2},
    "l1_regularization": {
        "problem_type": "regression",
        "train_test_split_params": {"test_size": test_size},
        "logspace_search": {"start": -3, "stop": 3, "num": 20, "base": 10},
        # tolerance over minimum error with which to search for the best model
        "error_tolerance_pct": 0.05,
        # minimum features to keep in final selection
        "min_feats_to_keep": 4,
        "random_seed": RANDOM_SEED,
    },
}

In [ ]:
selected_feats, df_fs = run_feature_selection_steps(
    # use only training data to avoid bias in test results
    X=X_train_all,
    y=y_train,
    fs_steps=fs_steps
)

In [ ]:
# build model input datasets
X_train = X_train_all[selected_feats]
X_test = X_test_all[selected_feats]

### Correlation check


In [ ]:
display(
    plot_correlation_matrix(
        # use only training data to avoid bias in test results
        df=df_input_train[selected_feats + [target_col]], method="pearson", fig_height=5
    )
)

### Multicollinearity check

In [ ]:
# compute the Variance Inflation Factor (VIF) for each feature
df_vif = pd.DataFrame(
    data=[variance_inflation_factor(X_train.values, i) for i in range(len(selected_feats))],
    index=selected_feats,
    columns=["VIF"]
).sort_values("VIF", ascending=False)

df_vif

## 5. Regression Model

### Select regressor: Linear Regression or XGBoost

In [ ]:
MODEL_SELECTION = "linear_regression"
# MODEL_SELECTION = "xgboost"

model_selection_error = ValueError(
    "'MODEL_SELECTION' must be either 'linear_regression' or 'xgboost'. "
    f"Got {MODEL_SELECTION} instead."
)

### Hyperparameter tuning with K-Fold Cross Validation

For a detailed explanation of XGBoost's parameters, refer to: https://www.kaggle.com/code/prashant111/a-guide-on-xgboost-hyperparameters-tuning/notebook

In [ ]:
if MODEL_SELECTION == "linear_regression":
    # ElasticNet is a linear regression model with combined L1 (Lasso)
    # and L2 (Ridge) priors as regularizer
    Estimator = ElasticNet
    cv_search_space = {
        "alpha": np.logspace(-4, 1, num=11, base=10.0), # 10e-4 to 10 in 11 steps
        "l1_ratio": np.linspace(0,1,9), # 0%, 12.5%, 25%, ... 100%
        "fit_intercept": [True],
        "max_iter": [2000], # use 2000 instead of defalult 1000
    }
elif MODEL_SELECTION == "xgboost":
    Estimator = XGBRegressor
    cv_search_space = {
        "objective": ["reg:squarederror"],
        "n_estimators": [20, 35, 50],
        "learning_rate": [0.1],
        "max_depth": [3, 4, 6],
        "min_child_weight": [2, 4],
        "gamma": [0, 0.5],
        "alpha": [0, 0.3],
        "scale_pos_weight": [1],
        "lambda": [1],
        ## "subsample": [0.8, 1.0],
        ## "colsample_bytree": [0.8, 1.0],
        "verbosity": [0],
    }
else:
    raise model_selection_error

For the full list of scikit-learn's scoring string names, refer to: https://scikit-learn.org/stable/modules/model_evaluation.html#string-name-scorers

In [ ]:
cv_scoring_metrics = {
    "neg_mean_absolute_error": "Mean Absolute Error",
    "neg_median_absolute_error": "Median Absolute Error",
    "neg_mean_squared_error": "Mean Squared Error",
    "neg_root_mean_squared_error": "Root Mean Squared Error",
    "max_error": "Maximum Residual Error",
    "r2": "R-squared (Coefficient of Determination)",
}
refit_metric = "neg_root_mean_squared_error"  # metric to optimize for the final model

In [ ]:
%%time
# define evaluation
kfold_cv = RepeatedKFold(n_splits=3, n_repeats=1, random_state=RANDOM_SEED)
# define search
grid_search = GridSearchCV(
    estimator=Estimator(),
    param_grid=cv_search_space,
    scoring=list(cv_scoring_metrics.keys()),
    cv=kfold_cv,
    refit=refit_metric,
    verbose=1,
)
# execute search
with warnings.catch_warnings(action="ignore"):
    result_cv = grid_search.fit(X_train, y_train)

In [ ]:
print("Grid Search CV Best Model - Scoring Metrics:")
for i, (metric_key, metric_name) in enumerate(cv_scoring_metrics.items(), start=1):
    print(
        f"  {str(i) + ".":>2} {metric_name:.<42} "
        f"{result_cv.cv_results_[f"mean_test_{metric_key}"][result_cv.best_index_]:+,.3f}"
    )
print(f"\nBest Hyperparameters: {result_cv.best_params_}")

### Final Model

In [ ]:
# instantiate model with best hyperparameters and additional kwargs
if MODEL_SELECTION == "linear_regression":
    model_kwargs = dict()
    model_fit_kwargs = dict()
elif MODEL_SELECTION == "xgboost":
    eval_metrics = dict(
        rmse="Root Mean Squared Error",
        mae="Mean Absolute Error",
        mape="Mean Absolute Percentage Error",
    )
    model_kwargs = dict(eval_metric=list(eval_metrics.keys()))
    model_fit_kwargs = dict(
        eval_set=[(X_train, y_train), (X_test, y_test)],
        verbose=False
    )
else:
    raise model_selection_error
    
model = Estimator(**result_cv.best_params_, **model_kwargs, random_state=RANDOM_SEED)

In [ ]:
# Fit model and make predictions
model.fit(X_train, y_train, **model_fit_kwargs)
# Make predictions
y_pred_train = pd.Series(data=model.predict(X_train), index=X_train.index)
y_pred = pd.Series(data=model.predict(X_test), index=X_test.index)

In [ ]:
if MODEL_SELECTION == "xgboost":
    display(plot_eval_metrics_xgb(model.evals_result(), eval_metrics))

### Feature Importance

- For Linear Regression: coefficients values and statistical significance
- For XGBoost: SHAP analysis and Gain Metric

In [ ]:
if MODEL_SELECTION == "linear_regression":
    df_coefficients = build_coefficients_table(
        coefficients=model.coef_,
        intercept=model.intercept_,
        X_train=X_train,
        y_pred_train=y_pred_train,
        y_train=y_train,
        problem_type="regression",
    )
    display(plot_coefficients_values(df_coefficients))
    display(plot_coefficients_significance(df_coefficients, log_scale=False))
    
elif MODEL_SELECTION == "xgboost":
    # compute SHAP values
    explainer = shap.Explainer(model)
    shap_values = explainer(X_test)
    # shap plots
    display(plot_shap_importance(shap_values))
    display(plot_shap_beeswarm(shap_values))
    display(plot_gain_metric_xgb(model, X_test))

else:
    raise model_selection_error

### Performance Metrics

In [ ]:
df_train_metrics = pd.Series(
    compute_regression_metrics(y_train, y_pred_train)
).to_frame(name="Train Metrics")
df_test_metrics = pd.Series(
    compute_regression_metrics(y_test, y_pred)
).to_frame(name="Test Metrics")

print("Final Model - Scoring Metrics on Train & Test Datasets:")
df_metrics = df_train_metrics.join(df_test_metrics)
display(df_metrics)